# Module 9: Reconcile Accounting Differences and Build Expanded Next-Year Pairs

本模块把 Module 8 的年度工作表，转为“本期库存信号 → 下一年度财务结果”的扩大配对样本。
仍然是 8 个 Code Cells：中文讲解，英文代码、注释、列名和文件名；不使用自定义财务处理函数。
不下载数据，不需要 SEC_HEADERS，也不需要安装新的包。

**实际操作顺序：**
1. 先保存现有 Notebook，再在 Module 8 后增加一个 Markdown 标题和下面的 8 个 Code Cells。
2. 可以直接复制本 Guide 的 Markdown 与代码；也可以单独运行 Guide，使用前面相同的项目目录。
3. Cell 1 查看路径。你之前的实际项目目录为 `[project root]`，不是 Downloads。
4. Cell 2–8 从上到下运行，不要只运行最后一格。本模块使用新变量 annual / paired，全部从已保存文件重建。
5. 保存 Notebook。重点检查 Cell 3 的收入调整、Cell 7 的样本统计以及 Cell 8 的保存位置。

输入：expanded_annual_audit.csv、expanded_company_readiness.csv、candidate_scope_review.csv、expanded_module8_metadata.json。
不要拿已删过年份的毛利/现金流工作表来 shift；这里从保留所有年度的完整审核表开始。
本模块不读取或应用 expanded_metric_review_decisions.csv 的人工金额，不用先填完那 150 条建议。
这些缺失仍留空；今后采用人工批准金额时，要另做有来源记录的导入步骤。

核心问题：**存货增长率比收入增长率高，是否与下一年度毛利率或经营现金流收入比下降有关？**
先建立可检查的配对，不在本模块计算显著性、模型或买卖建议。
产出仍是 provisional research pairs：42 家公司的行业资格和全部历史会计口径尚未最终审核。
年报提交时间通常晚于下一财年开始；本项目目前检验回顾性的年度关联，并非已验证的完整全年提前预测。


## Cell 1: Set Up the Project

Path.cwd() 读取运行环境的工作目录，不一定是 Notebook 所在目录。
优先复用前面已经设置的 PROJECT_ROOT，因此接在旧 Notebook 后面时一般不用改路径。
如果单独打开 Guide 后打印路径不对，把路径设置行改为：`PROJECT_ROOT = Path("[project root]")`。
PROCESSED_DIR 指向之前已保存 CSV 的目录，不用把文件搬到 Downloads，也不用重新运行下载模块。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)


## Cell 2: Read the Complete Annual History

annual 是 Module 8 完整审核表的新副本，当前缓存应为 656 行、42 家公司。
parse_dates 把字符串日期恢复为日期对象，后面才能减出相隔天数。
sort_values 保证每家公司按实际年度终点排序；reset_index 重新生成行号。
assert 检查公司与年度终点的组合是否重复：一个公司年度出现两行，会让“上一行/下一行”含义出错。

readiness 保留 Module 8 的每种结果至少 5 个可用年度的覆盖门槛。
company_review 重新读取人工公司表，尊重当前 exclude 决定，不把 pending 自动改成 include。
subgroup 从公司表带回，因为年度审核表本身没有这个列。
module8_metadata 把上游的数据版本、标签与处理方法带到本模块的最终说明文件。

打印数字只用于确认读到了正确文件；没有把 656 写成程序必须满足的硬性要求。
如果以后缓存更新，年度记录数可以变化。


In [ ]:
annual = pd.read_csv(
    PROCESSED_DIR / "expanded_annual_audit.csv",
    parse_dates=["period_start", "period_end", "filed"],
).sort_values(["ticker", "period_end"]).reset_index(drop=True)
assert not annual.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year rows."
readiness = pd.read_csv(PROCESSED_DIR / "expanded_company_readiness.csv")
company_review = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", keep_default_na=False)
annual = annual.merge(
    company_review[["ticker", "subgroup", "accounting_review_status", "include_decision"]],
    on="ticker", how="left", validate="many_to_one",
)
module8_metadata = json.loads(
    (PROCESSED_DIR / "expanded_module8_metadata.json").read_text(encoding="utf-8")
)
print("Annual rows:", len(annual))
print("Candidate companies:", annual["ticker"].nunique())
display(annual[["ticker", "period_end", "revenue", "gross_profit", "operating_cash_flow"]].head())


## Cell 3: Reconcile the 13 Identified Accounting Differences

这一格先解决前面已标出的 13 个差额，不能统一把它们当成数据错误。

**DDS 的 4 个年度：收入选到了总收入，而毛利对应 Net sales。**
2020-02-01、2021-01-30、2022-01-29、2023-01-28 的收入包含 Service charges and other income。
把独立核对的 Net sales 金额写入 revenue_for_analysis，后面的收入增长、毛利率、现金流收入比都用这一列。
原 revenue 不覆盖，避免失去 SEC 标签原始金额。DDS 的净销售额仍含建筑业务，不等于纯零售销售。
其余年度沿用现有收入，完整历史资格仍待审核；本次只修正明确识别的 4 条口径混用。

**BBY 的 3 条、URBN 的 6 条：毛利之前还单列了一项费用。**
BBY 是 Restructuring charges — cost of goods sold；URBN 是 Store impairment，或 Store impairment and lease abandonment charges。
保留已披露 gross_profit 和 cost_of_sales，将该项费用单独记为 extra_gross_profit_charge。
毛利已经扣过这些费用，不能再减一次。URBN 单列在毛利之后的 Asset impairment 也不能拿来代替毛利前门店费用。

构建顺序：建立证据表 → 按 ticker / period_end / accn 合并 → 新建分析收入 → 计算调整后剩余差额。
reported_net_sales、extra_gross_profit_charge 都来自报表项目，不是用差额倒算；金额单位均为 USD。
大金额中的 `_` 只帮助阅读，例如 `24_000_000` 与 `24000000` 相同。
部分 URBN 证据来自后续报告的比较栏或 ARS 年报，表中记录真实来源；它不证明该信息当时首次公开的时间。

remaining_bridge_gap = 分析收入 - 原成本 - 已核对单列费用 - 原毛利。
fillna(0) 只用于“没有登记额外桥接费用”的计算，不填收入、成本或毛利的缺失值。
bridge_unresolved 仅标记已知非零残差；残差为空、或残差为零，都不等于整体会计口径已审核。
两个 assert 分别防止证据没有匹配到文件、以及报表数字更新后仍误套旧修正。不是自动把差额强制清零。

预计打印：Evidence matches = 13；Revenue basis adjustments = 4；Unresolved nonzero bridge gaps = 0。

核对来源：[DDS 2020](https://www.sec.gov/Archives/edgar/data/28917/000002891720000041/dds-02012020x10k.htm)、[DDS 2021](https://www.sec.gov/Archives/edgar/data/28917/000002891721000066/dds-20210130.htm)、[DDS 2022](https://www.sec.gov/Archives/edgar/data/28917/000002891722000009/dds-20220129.htm)、[DDS 2023](https://www.sec.gov/Archives/edgar/data/28917/000002891723000007/R4.htm)。
费用来源：[BBY 2011](https://www.sec.gov/Archives/edgar/data/764478/000104746911004045/a2203505z10-k.htm)、[BBY 2012](https://www.sec.gov/Archives/edgar/data/764478/000076447812000035/bby-201210k.htm)、[BBY 2016](https://www.sec.gov/Archives/edgar/data/764478/000076447816000064/bby-2016x10k.htm)、[URBN 2021 比较栏](https://www.sec.gov/Archives/edgar/data/912615/000156459021017306/urbn-10k_20210131.htm)、[URBN 2025 比较栏](https://www.sec.gov/Archives/edgar/data/912615/000095017025048506/urbn-20250131.htm)、[URBN 2026 ARS](https://www.sec.gov/Archives/edgar/data/912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf)。


In [ ]:
sec_archive = "https://www.sec.gov/Archives/edgar/data/"
evidence = pd.DataFrame([
    ["BBY", "2011-02-26", "0001047469-11-004045", None, 24_000_000, "764478/000104746911004045/a2203505z10-k.htm", "Restructuring charges in cost of goods sold"],
    ["BBY", "2012-03-03", "0000764478-12-000035", None, 19_000_000, "764478/000076447812000035/bby-201210k.htm", "Restructuring charges in cost of goods sold"],
    ["BBY", "2016-01-30", "0000764478-16-000064", None, 3_000_000, "764478/000076447816000064/bby-2016x10k.htm", "Restructuring charges in cost of goods sold"],
    ["DDS", "2020-02-01", "0000028917-20-000041", 6_203_520_000, 0, "28917/000002891720000041/dds-02012020x10k.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2021-01-30", "0000028917-21-000066", 4_300_895_000, 0, "28917/000002891721000066/dds-20210130.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2022-01-29", "0000028917-22-000009", 6_492_993_000, 0, "28917/000002891722000009/dds-20220129.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2023-01-28", "0000028917-23-000007", 6_871_081_000, 0, "28917/000002891723000007/R4.htm", "Net sales exclude service charges and other income"],
    ["URBN", "2020-01-31", "0001564590-20-014397", None, 14_611_000, "912615/000156459021017306/urbn-10k_20210131.htm", "Store impairment; comparative column in 2021 filing"],
    ["URBN", "2021-01-31", "0001564590-21-017306", None, 15_496_000, "912615/000156459021017306/urbn-10k_20210131.htm", "Store impairment"],
    ["URBN", "2023-01-31", "0000950170-23-011597", None, 6_417_000, "912615/000095017025048506/urbn-20250131.htm", "Store charges; comparative column in 2025 filing"],
    ["URBN", "2024-01-31", "0000950170-24-039320", None, 11_875_000, "912615/000095017025048506/urbn-20250131.htm", "Store charges; comparative column in 2025 filing"],
    ["URBN", "2025-01-31", "0000950170-25-048506", None, 4_601_000, "912615/000095017025048506/urbn-20250131.htm", "Store impairment and lease abandonment charges"],
    ["URBN", "2026-01-31", "0001193125-26-137916", None, 1_989_000, "912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf", "Store charges; matching annual report in a separate ARS filing"],
], columns=["ticker", "period_end", "accn", "reported_net_sales", "extra_gross_profit_charge", "evidence_url", "evidence_note"])
evidence["period_end"] = pd.to_datetime(evidence["period_end"])
evidence["evidence_url"] = sec_archive + evidence["evidence_url"]
annual = annual.merge(evidence, on=["ticker", "period_end", "accn"], how="left", validate="one_to_one")
annual["revenue_for_analysis"] = annual["reported_net_sales"].fillna(annual["revenue"])
annual["revenue_basis_adjusted"] = annual["reported_net_sales"].notna()
annual["remaining_bridge_gap"] = (
    annual["revenue_for_analysis"] - annual["cost_of_sales"]
    - annual["extra_gross_profit_charge"].fillna(0) - annual["gross_profit"]
)
annual["bridge_unresolved"] = annual["remaining_bridge_gap"].abs() > 1
matched = annual["evidence_url"].notna()
assert matched.sum() == len(evidence), "An accounting evidence row did not match the selected filing."
assert annual.loc[matched, "remaining_bridge_gap"].abs().le(1).all(), "Review the accounting reconciliation."
print("Evidence matches:", int(matched.sum()))
print("Revenue basis adjustments:", int(annual["revenue_basis_adjusted"].sum()))
print("Unresolved nonzero bridge gaps:", int(annual["bridge_unresolved"].sum()))
display(annual.loc[annual["revenue_basis_adjusted"], ["ticker", "period_end", "revenue", "revenue_for_analysis"]])


## Cell 4: Recalculate Ratios and Set Provisional Coverage Rules

调整收入后，不能继续直接用 Module 8 保存的比率，否则分母仍是旧收入。
因此重算 gross_margin = gross_profit / revenue_for_analysis，以及 ocf_margin = operating_cash_flow / revenue_for_analysis。
base_values_ok 重新判断收入与资产为正、存货非负且不超过资产；负毛利、负现金流本身允许存在。
GM 额外要求毛利存在、没有未解释的非零桥接差；OCF 只要求现金流存在，不要求成本或毛利存在。
成本缺失但毛利已直接披露时，仍可以计算毛利率，整体历史口径仍是 pending。

BBY 早期历史有多次终止经营及重列。2012 报表把此前未单独列示的业务改为终止经营；2016 报表也说明 Europe、mindSHIFT、Five Star 的历史比较数重列。
本次没有把这些年的收入、库存与现金流统一重建，所以 **BBY 的 period_end 早于 2017-01-01 的记录暂缓参与配对**。
historical_scope_hold 表示需要进一步统一历史口径，不表示原数据全错；日期只是本模块保守的工作边界，不代表 2017 年以后全部已核实。
上一期若处于 hold，本期增长也不采用，因此边界后的第一年不会硬接到暂缓年份。
其余公司沿用 Module 8 的已知断点；不声称它们没有其他未发现的业务变化。

gm_coverage_ok / ocf_coverage_ok 使用 Module 8 的至少 5 个可用年度门槛，并尊重 exclude。
set_index("ticker") 让公司名成为查找键；map 根据本行 ticker 找到该公司的年度数量。
pending 公司可进入暂定研究配对，状态随数据保存，不会被升级成正式合格公司。
这不是要求每家公司至少有 5 个最终配对；配对数将在 Cell 7 单独统计。

来源：[BBY 2012 历史重列](https://www.sec.gov/Archives/edgar/data/764478/000076447812000035/bby-201210k.htm)、[BBY 2016 终止经营说明](https://www.sec.gov/Archives/edgar/data/764478/000076447816000064/bby-2016x10k.htm)。


In [ ]:
annual["base_values_ok"] = (
    annual["revenue_for_analysis"].gt(0) & annual["total_assets"].gt(0)
    & annual["inventory"].ge(0) & annual["inventory"].le(annual["total_assets"])
)
annual["gm_row_usable"] = annual["base_values_ok"] & annual["gross_profit"].notna() & ~annual["bridge_unresolved"]
annual["ocf_row_usable"] = annual["base_values_ok"] & annual["operating_cash_flow"].notna()
annual["gross_margin"] = annual["gross_profit"] / annual["revenue_for_analysis"]
annual["ocf_margin"] = annual["operating_cash_flow"] / annual["revenue_for_analysis"]
annual.loc[~annual["gm_row_usable"], "gross_margin"] = float("nan")
annual.loc[~annual["ocf_row_usable"], "ocf_margin"] = float("nan")
annual["historical_scope_hold"] = annual["ticker"].eq("BBY") & annual["period_end"].lt(pd.Timestamp("2017-01-01"))
coverage = readiness.set_index("ticker")
company_allowed = annual["include_decision"].isin(["pending", "include"])
annual["gm_coverage_ok"] = annual["ticker"].map(coverage["gm_usable_years"]).ge(5) & company_allowed
annual["ocf_coverage_ok"] = annual["ticker"].map(coverage["ocf_usable_years"]).ge(5) & company_allowed
print("Historical rows held for further scope review:", int(annual["historical_scope_hold"].sum()))


## Cell 5: Attach Previous and Next Annual Records

这一格只排列时间，不计算研究结论，也不先删除缺失年份。
annual 按 ticker / period_end 排序后，paired 复制全部行。
groupby("ticker") 将每家公司单独处理，避免把另一家公司的年度当成前后年度。
shift(1) 取上一行；shift(-1) 取下一行。它们还不保证真隔一年，下一格要检查连续性。
循环只是把相同操作应用到多列，不用写十多遍类似代码。

例如当前是 2022 年、下一行是 2024 年：shift 会先带来 2024 年，Cell 6 会发现隔了约两年并拒绝配对。
如果先删掉 2023 年的缺失记录，再 shift，就会掩盖中间缺失或口径变化；这里保留整个时间轴。
prev_ 列用于计算本期增长，next_ 比率用于计算结果。
next_filed / next_accn 用于追溯目标年报，不能当作本期模型特征。


In [ ]:
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
paired = annual.copy()
for column in [
    "period_end", "period_days", "revenue_for_analysis", "inventory",
    "base_values_ok", "known_scope_break", "historical_scope_hold",
]:
    paired["prev_" + column] = annual.groupby("ticker")[column].shift(1)
for column in [
    "period_start", "period_end", "period_days", "filed", "accn", "gross_margin", "ocf_margin",
    "gm_row_usable", "ocf_row_usable", "known_scope_break", "historical_scope_hold",
]:
    paired["next_" + column] = annual.groupby("ticker")[column].shift(-1)
display(paired[["ticker", "prev_period_end", "period_end", "next_period_end"]].head(8))


## Cell 6: Build Inventory Signals and Validate Next-Year Pairs

**先判断上一年度能否用来计算增长。**
两个年度终点相隔 350–380 天，且本年度 start 等于上一年度 end 的下一天，才是连续年度。
base_values_ok 要求本期和上一期基础金额可用；上一年收入、存货必须大于零，防止除零。
本期存货可以为零，其增长率会是 -100%，不因此删掉。
本期和上一期都不能是已知口径变化年或 BBY 暂缓年。这是保守规则，也防止收购当年与下一完整年度直接比较。

revenue_growth = 本期分析收入 / 上期分析收入 - 1。
inventory_growth = 本期存货 / 上期存货 - 1。
inventory_sales_gap = 存货增长率 - 收入增长率。
例如 0.20 - 0.08 = 0.12，意味着存货增长高出收入增长 12 个百分点。
gap 为正不一定表示存货金额增加：收入跌 20%、存货跌 5%，gap 仍是 +15 个百分点。
feature_usable 不通过时，三个数留空；inventory_outpaced_sales 也保留 pd.NA，而不是误记为 False。

**再判断下一年度能否作为结果。**
同样检查年度终点与 start 的连续性。本期 filed 不早于本期 end，且必须早于目标年度 end。
这个时间条件只说明年报在目标年度结束前提交，不代表在目标年度开始前已知。
common_pair_ok 组合共同规则，再分别加上毛利或现金流的本期、下一期可用条件以及公司覆盖门槛。
所以 OCF 配对不会因为毛利缺失而被删；也不会为了让 GM 样本变大而把缺失毛利统一推导。


In [ ]:
previous_end_gap = (paired["period_end"] - paired["prev_period_end"]).dt.days
previous_start_gap = (paired["period_start"] - paired["prev_period_end"]).dt.days
paired["consecutive_previous_year"] = previous_end_gap.between(350, 380) & previous_start_gap.eq(1)
paired["feature_usable"] = (
    paired["consecutive_previous_year"] & paired["base_values_ok"] & paired["prev_base_values_ok"].eq(True)
    & paired["prev_revenue_for_analysis"].gt(0) & paired["prev_inventory"].gt(0)
    & ~paired["known_scope_break"] & paired["prev_known_scope_break"].eq(False)
    & ~paired["historical_scope_hold"] & paired["prev_historical_scope_hold"].eq(False)
)
paired["revenue_growth"] = paired["revenue_for_analysis"] / paired["prev_revenue_for_analysis"] - 1
paired["inventory_growth"] = paired["inventory"] / paired["prev_inventory"] - 1
paired["inventory_sales_gap"] = paired["inventory_growth"] - paired["revenue_growth"]
paired.loc[~paired["feature_usable"], ["revenue_growth", "inventory_growth", "inventory_sales_gap"]] = float("nan")
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"].gt(0).astype("boolean")
paired.loc[~paired["feature_usable"], "inventory_outpaced_sales"] = pd.NA
next_end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
next_start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = next_end_gap.between(350, 380) & next_start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (paired["filed"] >= paired["period_end"]) & (paired["filed"] < paired["next_period_end"])
common_pair_ok = (
    paired["feature_usable"] & paired["consecutive_next_year"] & paired["filed_time_ok"]
    & paired["next_known_scope_break"].eq(False) & paired["next_historical_scope_hold"].eq(False)
)
paired["gm_pair_usable"] = common_pair_ok & paired["gm_coverage_ok"] & paired["gm_row_usable"] & paired["next_gm_row_usable"].eq(True)
paired["ocf_pair_usable"] = common_pair_ok & paired["ocf_coverage_ok"] & paired["ocf_row_usable"] & paired["next_ocf_row_usable"].eq(True)
print("Usable inventory feature rows:", int(paired["feature_usable"].sum()))


## Cell 7: Calculate Outcomes and Inspect the Expanded Samples

next_gross_margin_change = 下一年度毛利率 - 本年度毛利率。
next_ocf_margin_change = 下一年度现金流收入比 - 本年度现金流收入比。
例如 -0.03 表示下降 3 个百分点，不是金额少了 3 美元，也不是毛利率相对跌了 3%。
现金流收入比下降不等于现金流绝对金额一定下降。

循环按各自 pair_usable 清空不合格结果；下降标签用可缺失 boolean 类型。
因此最后一年没有下一期，会留下 NaN / pd.NA，不会被当成“下一年没有下降”。

available_before_target_start 记录本期年报是否在目标年度开始前提交。
days_after_target_start 是本期 filed 减去目标年度 start 的日历天差，正数表示目标年度已开始。
period_length_changed 看本期与上期长度，pair_period_length_changed 还考虑目标年度长度。
350–380 天范围可容纳普通年度和 52/53 周财年，长度差超过 3 天就做敏感性提醒，不自动按天数缩放收入。
含目标年度长度的标记只用于事后样本审核，不能拿来当作当时已知的模型特征。

gm_pairs / ocf_pairs 是分别筛选出的暂定研究表；pair_summary 列出每家公司各有多少个配对。
显示的是 company-year pairs：一家公司可以贡献多个年度，并非同样数量的独立公司。
后续统计应考虑同一公司多年相关、不同子行业及共同年度冲击；不能把配对行数直接当作独立样本量。

核对重点：两个研究表不应为空；OCF 公司数可能更多；最新年度应没有目标；提交日期范围应合理。
使用你当前未更新的缓存与 pending 公司表，预计 GM 为 36 家 / 401 个配对，OCF 为 42 家 / 490 个配对。
538 行可计算库存特征；两个样本的提交日期均晚于目标年度开始，天差为 38–134 天。
这些是当前版本的验证参照，今后补数据、核对业务口径或修改人工纳入决定，数量可以变化。
审核统计中的条件可以重叠，例如同一行既没有上一期，也没有下一期，不应把这些计数相加当成总删样本数。


In [ ]:
paired["next_gross_margin_change"] = paired["next_gross_margin"] - paired["gross_margin"]
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
for outcome, label, usable in [
    ("next_gross_margin_change", "gross_margin_declined", "gm_pair_usable"),
    ("next_ocf_margin_change", "ocf_margin_declined", "ocf_pair_usable"),
]:
    paired.loc[~paired[usable], outcome] = float("nan")
    paired[label] = paired[outcome].lt(0).astype("boolean")
    paired.loc[~paired[usable], label] = pd.NA
paired["available_before_target_start"] = paired["filed"] < paired["next_period_start"]
paired["days_after_target_start"] = (paired["filed"] - paired["next_period_start"]).dt.days
paired["period_length_changed"] = (paired["period_days"] - paired["prev_period_days"]).abs() > 3
paired["pair_period_length_changed"] = paired["period_length_changed"] | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
pair_summary = paired.groupby(["ticker", "subgroup"], sort=False)[["gm_pair_usable", "ocf_pair_usable"]].sum()
pair_summary = pair_summary.rename(columns={"gm_pair_usable": "gm_pairs", "ocf_pair_usable": "ocf_pairs"}).reset_index()
for name, table in [("Gross margin", gm_pairs), ("Operating cash flow", ocf_pairs)]:
    print(name, "companies:", table["ticker"].nunique(), "| pairs:", len(table))
    print("Available before target year starts:", int(table["available_before_target_start"].sum()))
    print("Pairs with different annual lengths:", int(table["pair_period_length_changed"].sum()))
    display(table["days_after_target_start"].agg(["min", "max"]).to_frame("Days After Target Start"))
display(pair_summary)
display(paired[["feature_usable", "consecutive_previous_year", "consecutive_next_year", "gm_pair_usable", "ocf_pair_usable"]].sum().rename("Rows Passing").to_frame())


## Cell 8: Save the Pairing Audit and Research Tables

这格保存的是新的 Module 9 文件，不覆盖 Module 8 的输入或人工审核文件。
expanded_module9_annual_audit.csv：完整年度表，保留原收入、分析收入、证据、暂缓标记。
expanded_module9_pairing_audit.csv：全部年度及前后期、通过条件、特征、目标，供查失败原因。
expanded_gm_next_year_pairs.csv / expanded_ocf_next_year_pairs.csv：下个模块实际读取的两个暂定研究表。
expanded_next_year_pair_counts.csv：公司和子行业的配对数量。
expanded_accounting_bridge_evidence.csv：13 条具体来源与修正金额。
expanded_module9_metadata.json：上游版本、本模块规则、样本数量，以及尚未完成的审核状态。

outputs 是“文件名 → DataFrame”的字典，循环对每个表调用 to_csv 保存，index=False 避免把行号写成多余一列。
date_format 统一日期格式。JSON 的 indent=2 便于直接阅读处理说明。
metadata 记录 accounting_scope_verified=False、point_in_time_verified=False，防止后续把暂定表误称为最终已审核样本。

运行完成后保存你的 Notebook。下一步再开展扩大样本的分组比较、相关性和敏感性分析。
Report 的研究方法部分应说明：分开的结果样本、DDS 收入口径调整、BBY 早期历史暂缓、已知断点处理、52/53 周提醒及提交日期局限。
这些规则先于结果分析确定，不应在看见结论后改动来强化关系。


In [ ]:
outputs = {
    "expanded_module9_annual_audit.csv": annual,
    "expanded_module9_pairing_audit.csv": paired,
    "expanded_gm_next_year_pairs.csv": gm_pairs,
    "expanded_ocf_next_year_pairs.csv": ocf_pairs,
    "expanded_next_year_pair_counts.csv": pair_summary,
    "expanded_accounting_bridge_evidence.csv": evidence,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module8_metadata": module8_metadata,
    "annual_rows": len(annual), "accounting_evidence_rows": len(evidence),
    "revenue_basis_adjustments": int(annual["revenue_basis_adjusted"].sum()),
    "gm_companies": gm_pairs["ticker"].nunique(), "gm_pairs": len(gm_pairs),
    "ocf_companies": ocf_pairs["ticker"].nunique(), "ocf_pairs": len(ocf_pairs),
    "sample_status": "provisional", "min_annual_coverage": 5, "manual_metric_decisions_applied": False,
    "unit": "USD; ratios and growth rates stored as decimals",
    "revenue_rule": "Use verified net sales for the four identified DDS rows; preserve original revenue",
    "bridge_rule": "Preserve reported gross profit; record independently verified extra charges separately",
    "scope_rule": "Exclude features or targets involving known breaks; hold BBY period ends before 2017-01-01",
    "historical_scope_hold_sources": [
        "https://www.sec.gov/Archives/edgar/data/764478/000076447812000035/bby-201210k.htm",
        "https://www.sec.gov/Archives/edgar/data/764478/000076447816000064/bby-2016x10k.htm",
    ],
    "continuity_rule": "Adjacent annual ends 350-380 days apart and the next start one day after the prior end",
    "timing_rule": "Current filing date is a proxy; it must precede target end, but may follow target start",
    "research_design": "Retrospective annual association; current and target ratios both required for change outcomes",
    "accounting_scope_verified": False, "point_in_time_verified": False, "next_year_pairs_built": True,
}
(PROCESSED_DIR / "expanded_module9_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 9 outputs saved in:", PROCESSED_DIR)
